## PDF Analysis with Claude Managed Agents

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")
session_id = os.getenv("SESSION_ID")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Upload PDF using the Files API

In [ ]:
with open("./Docs/margies-travel.pdf", "rb") as f:
    file_upload = client.beta.files.upload(file=("margies-travel.pdf", f, "application/pdf"))

### Create an Agent

In [ ]:
agent = client.beta.agents.create(
    name="PDF-Analysis-Agent",
    model=claude_model_name,
    system="You are a helpful AI Assistant.",
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

### Execute the Agent

In [ ]:
with client.beta.sessions.events.stream(session_id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session_id,
                betas = ["files-api-2025-04-14"],
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                              "type": "document",
                              "source": {"type": "file", "file_id": file_upload.id}    
                            },
                            {
                                "type": "text",
                                "text": """Tell me something about Margie's Travel. Also analyze the images 
                                           present in the PDF Document""",
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break